# 05 病灶自动勾画

这本 Notebook 将三序列 MRI 和人工病灶标注整理成 nnU-Net 数据集，完成五折分割训练、预测和分割质量评价。训练得到的自动掩模还可以交给 06，观察替换分类模型中的人工掩模后，预测表现怎样变化。

开始前需要完成 00，并在配置中填写标签 1、2、3 的真实名称。确认人工标注覆盖完整三维体积后，将 `annotation_scope` 设为 `full_volume`；未标注区域若被当成背景，会给分割训练引入错误标签。

训练在已准备好 nnU-Net 的服务器上进行。当前配置使用 `2d`，是否适合实际层厚和病灶形态，需要结合影像检查结果判断。下面先读取配置文件。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg


def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics", "explain": root / "explain", "segmentation": root / "segmentation",
        "reports": root / "reports",
    }


def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")


def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result


def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "original_resnet_integration_no_architecture_optimization", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)


def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table


def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)


def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)


def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")
    if manifest["notebook_definitions_sha256"] != source_snapshot(cfg):
        raise ValueError("Notebook 定义与数据包导出时不同，请使用对应版本或重新导出。")


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本数据包只有分类缓存，没有原分辨率影像；影像组学和分割需另外准备原生数据。")


In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


## 从数据导出到分割评价

下面按分割实验的几个阶段定义函数，实际训练由 nnU-Net 完成。

| 函数 | 作用 |
| --- | --- |
| `locations`、`_volumes` | 整理分割目录，核对影像、标注和预测的空间信息 |
| `export_dataset` | 导出三序列影像、训练标签、数据集说明和固定五折划分 |
| `verify_dataset` | 核对患者清单、标签定义及训练标签目录 |
| `command_plan`、`run_phase` | 生成并分阶段执行预处理、训练与预测命令 |
| `install_splits` | 将已确定的五折划分放入 nnU-Net 预处理目录 |
| `evaluate_masks` | 在原生空间逐患者、逐标签计算 Dice 和 HD95 |
| `build_validation_mask_cache`、`verify_validation_mask_cache` | 把验证预测转成分类器的四层掩模，并记录、核对文件版本 |

这样可以分别检查数据导出、训练、分割效果和分类输入转换，发现问题时能够定位到具体阶段。


In [ ]:
"""New nnU-Net dataset/command integration and native-space evaluation."""
from __future__ import annotations

import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd



def locations(cfg):
    root = paths(cfg)["segmentation"]
    dataset_id = int(cfg["segmentation"]["dataset_id"])
    if not 100 <= dataset_id <= 999:
        raise ValueError("本项目使用100～999的分割数据集ID。")
    name = f"Dataset{dataset_id:03d}_PSCI"
    return {"root": root, "name": name, "raw": root / "nnUNet_raw" / name,
            "preprocessed": root / "nnUNet_preprocessed" / name,
            "results": root / "nnUNet_results", "validation": root / "validation_predictions",
            "oof": root / "development_oof_predictions"}


def _volumes(patient, predicted_path=None):
    import nibabel as nib
    image = nib.load(str(patient.t1))
    truth = nib.load(str(patient.manual_mask))
    others = [nib.load(str(patient[s])) for s in SEQUENCES[1:]] + [truth]
    if predicted_path is not None:
        others.append(nib.load(str(predicted_path)))
    if any(v.shape != image.shape or not np.allclose(v.affine, image.affine, atol=1e-4) for v in others):
        raise ValueError(f"{patient.mr}的影像、真值或预测几何不一致；不自动重采样修正。")
    for volume in others[-2:] if predicted_path is not None else [truth]:
        values = np.unique(np.asarray(volume.dataobj))
        if not np.isfinite(values).all() or not set(values) <= {0, 1, 2, 3}:
            raise ValueError(f"{patient.mr}标签不是0/1/2/3。")
    prediction = others[-1] if predicted_path is not None else None
    return image, truth, prediction


def export_dataset(cfg):
    require_native_images(cfg)
    if cfg["annotation_scope"] != "full_volume":
        raise ValueError("分割基线当前要求完整三维标注；先核对annotation_scope，部分标注不能直接当全背景。")
    names = cfg["label_names"]
    if set(names) != {"1", "2", "3"} or any(not isinstance(v, str) or not v.strip() for v in names.values()):
        raise ValueError("请按真实标注填写三个label_names，不能猜测病灶类型。")
    if len(set(names.values()) | {"background"}) != 4:
        raise ValueError("前景标签名称应唯一且不能命名为background。")
    loc, cohort = locations(cfg), read_cohort(cfg)
    if "annotation_scope" not in cohort or not cohort.annotation_scope.eq("full_volume").all():
        raise ValueError("患者清单中存在局部标注或未确认的病例；完整体积导出不能把未标注层当作背景。")
    fresh(loc["raw"])
    for _, patient in cohort.iterrows():
        _volumes(patient)
    train_images, train_labels, test_images = [loc["raw"] / s for s in ("imagesTr", "labelsTr", "imagesTs")]
    for folder in (train_images, train_labels, test_images):
        folder.mkdir(parents=True)
    for _, patient in cohort.iterrows():
        training = patient.split == "development"
        target = train_images if training else test_images
        for i, sequence in enumerate(SEQUENCES):
            shutil.copy2(patient[sequence], target / f"{patient.mr}_{i:04d}.nii.gz")
        if training:
            shutil.copy2(patient.manual_mask, train_labels / f"{patient.mr}.nii.gz")
    dev = cohort.query("split == 'development'")
    splits = []
    for fold in range(1, 6):
        valid_mrs = dev.loc[dev.development_fold == fold, "mr"].tolist()
        training_mrs = dev.loc[dev.development_fold != fold, "mr"].tolist()
        splits.append({"train": training_mrs, "val": valid_mrs})
        inputs = loc["root"] / "oof_inputs" / f"fold_{fold - 1}"
        inputs.mkdir(parents=True)
        for mr in valid_mrs:
            for i in range(3):
                filename = f"{mr}_{i:04d}.nii.gz"
                shutil.copy2(train_images / filename, inputs / filename)
    dump_json(loc["raw"] / "dataset.json", {"channel_names": {"0": "T1-FLAIR", "1": "T2", "2": "DWI"},
        "labels": {"background": 0, **{value: int(key) for key, value in names.items()}},
        "numTraining": len(dev), "file_ending": ".nii.gz"})
    dump_json(loc["root"] / "splits_final.json", splits)
    dump_json(loc["root"] / "dataset_provenance.json", {"cohort_sha256": digest(paths(cfg)["cohort"]),
        "validation_labels_in_training": False, "annotation_scope": cfg["annotation_scope"],
        "configuration": cfg["segmentation"]["configuration"], "label_names": names})
    record(cfg, "segmentation_dataset")
    return loc


def verify_dataset(cfg):
    loc, cohort = locations(cfg), read_cohort(cfg)
    meta = json.loads((loc["root"] / "dataset_provenance.json").read_text(encoding="utf-8"))
    if meta["cohort_sha256"] != digest(paths(cfg)["cohort"]):
        raise ValueError("分割导出后病例清单发生变化。")
    if meta["configuration"] != cfg["segmentation"]["configuration"] or meta["label_names"] != cfg["label_names"]:
        raise ValueError("分割配置或标签定义发生变化，请使用新的run_id导出。")
    expected = set(cohort.loc[cohort.split == "development", "mr"])
    actual = {p.name.removesuffix(".nii.gz") for p in (loc["raw"] / "labelsTr").glob("*.nii.gz")}
    if actual != expected:
        raise ValueError("labelsTr与开发队列不完全一致，禁止继续训练。")


def command_plan(cfg):
    loc = locations(cfg)
    settings = cfg["segmentation"]
    dataset, configuration = str(settings["dataset_id"]), settings["configuration"]
    if configuration not in ("2d", "3d_fullres"):
        raise ValueError("首版只提供2d或3d_fullres基线。")
    plan = {"preprocess": [["nnUNetv2_plan_and_preprocess", "-d", dataset, "--verify_dataset_integrity"]],
            "train": [], "predict_oof": [], "predict_validation": []}
    for fold in range(5):
        plan["train"].append(["nnUNetv2_train", dataset, configuration, str(fold), "--npz", "-device", settings["device"]])
        plan["predict_oof"].append(["nnUNetv2_predict", "-i", str(loc["root"] / "oof_inputs" / f"fold_{fold}"),
            "-o", str(loc["oof"] / f"fold_{fold}"), "-d", dataset, "-c", configuration, "-f", str(fold), "-device", settings["device"]])
    plan["predict_validation"] = [["nnUNetv2_predict", "-i", str(loc["raw"] / "imagesTs"),
        "-o", str(loc["validation"]), "-d", dataset, "-c", configuration, "-f", "0", "1", "2", "3", "4", "-device", settings["device"]]]
    return plan


def install_splits(cfg):
    verify_dataset(cfg)
    loc = locations(cfg)
    if not loc["preprocessed"].is_dir():
        raise FileNotFoundError("先在指定服务器完成preprocess，再安装固定splits_final.json。")
    source = loc["root"] / "splits_final.json"
    destination = loc["preprocessed"] / "splits_final.json"
    if destination.exists():
        if json.loads(destination.read_text()) != json.loads(source.read_text()):
            raise ValueError("已有nnU-Net划分与固定开发折不一致；不覆盖。")
    else:
        shutil.copy2(source, destination)
    return destination


def run_phase(cfg, phase):
    """Called only by an explicitly enabled server notebook cell; no shell=True."""
    verify_dataset(cfg)
    loc = locations(cfg)
    if phase not in command_plan(cfg):
        raise ValueError("未知分割阶段。")
    if phase != "preprocess":
        install_splits(cfg)
    env = os.environ.copy()
    env.update(nnUNet_raw=str(loc["raw"].parent), nnUNet_preprocessed=str(loc["preprocessed"].parent),
               nnUNet_results=str(loc["results"]))
    for folder in (loc["preprocessed"].parent, loc["results"]):
        folder.mkdir(parents=True, exist_ok=True)
    for command in command_plan(cfg)[phase]:
        print(shlex.join(command), flush=True)
        subprocess.run(command, check=True, env=env)
    record(cfg, "segmentation_" + phase)


def evaluate_masks(cfg, scope="validation"):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    verify_dataset(cfg)
    loc, cohort = locations(cfg), read_cohort(cfg)
    if scope not in ("validation", "development_oof"):
        raise ValueError("scope只能是validation或development_oof。")
    cases = cohort.query("split == 'validation'" if scope == "validation" else "split == 'development'")
    rows = []
    for _, patient in cases.iterrows():
        directory = loc["validation"] if scope == "validation" else loc["oof"] / f"fold_{int(patient.development_fold) - 1}"
        image, truth, prediction = _volumes(patient, directory / f"{patient.mr}.nii.gz")
        if image.header.get_xyzt_units()[0] != "mm":
            raise ValueError(f"{patient.mr}的NIfTI空间单位不是明确的mm；请核查后再报告HD95_mm。")
        true_values, pred_values = np.asarray(truth.dataobj), np.asarray(prediction.dataobj)
        for label in (1, 2, 3):
            a, b = true_values == label, pred_values == label
            na, nb = int(a.sum()), int(b.sum())
            row = {"mr": patient.mr, "scope": scope, "label": label, "true_voxels": na, "predicted_voxels": nb}
            if not na and not nb:
                row.update(dice=np.nan, hd95_mm=np.nan, status="both_empty_excluded_from_mean")
            elif not na or not nb:
                row.update(dice=0.0, hd95_mm=np.nan, status="one_empty_hd95_undefined")
            else:
                surface_a = a ^ binary_erosion(a, border_value=0)
                surface_b = b ^ binary_erosion(b, border_value=0)
                distances = np.concatenate([distance_transform_edt(~surface_b, sampling=image.header.get_zooms()[:3])[surface_a],
                                            distance_transform_edt(~surface_a, sampling=image.header.get_zooms()[:3])[surface_b]])
                row.update(dice=float(2 * (a & b).sum() / (na + nb)), hd95_mm=float(np.percentile(distances, 95)), status="ok")
            rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(loc["root"] / f"{scope}_segmentation_metrics.csv", index=False)
    summary = table.groupby("label").agg(n_cases=("mr", "size"), n_dice_defined=("dice", "count"),
        mean_dice=("dice", "mean"), n_hd95_defined=("hd95_mm", "count"), mean_hd95_mm=("hd95_mm", "mean"))
    summary.to_csv(loc["root"] / f"{scope}_segmentation_summary.csv")
    return table, summary


def build_validation_mask_cache(cfg):
    from scipy.ndimage import zoom
    verify_dataset(cfg)
    loc, p = locations(cfg), paths(cfg)
    target = loc["root"] / "validation_masks.npz"
    fresh(target)
    cases = read_cohort(cfg).query("split == 'validation'").reset_index(drop=True)
    result = np.zeros((len(cases), 4, 3, 224, 224), dtype=np.uint8)
    with np.load(p["cache"], allow_pickle=False) as cache:
        layers = cache["max_layers"]
        for i, patient in cases.iterrows():
            image, _, predicted = _volumes(patient, loc["validation"] / f"{patient.mr}.nii.gz")
            labels = np.asarray(predicted.dataobj)
            z0 = int(layers[int(patient.cache_index)]) - 1
            if z0 < 0 or z0 + 3 >= image.shape[2]:
                raise ValueError("自动掩模四层取层越界。")
            for s in range(4):
                for label in (1, 2, 3):
                    result[i, s, label - 1] = zoom((labels[:, :, z0 + s] == label).astype(np.uint8),
                        (224 / image.shape[0], 224 / image.shape[1]), order=0, prefilter=False)
    np.savez_compressed(target, mrs=cases.mr.to_numpy(dtype=str), masks=result)
    dump_json(loc["root"] / "automatic_input_note.json", {
        "scope": "validation_only", "experiment": "manual_model_auto_mask_substitution",
        "manual_slice_selection_still_required": True,
        "mask_cache_sha256": digest(target),
        "predictions_sha256": {mr: digest(loc["validation"] / f"{mr}.nii.gz") for mr in cases.mr},
        "not_implemented": "用嵌套交叉拟合自动掩模重训分类器的完整自动联合开发流程"})
    return target


def verify_validation_mask_cache(cfg):
    verify_dataset(cfg)
    loc = locations(cfg)
    metadata = json.loads((loc["root"] / "automatic_input_note.json").read_text(encoding="utf-8"))
    if digest(loc["root"] / "validation_masks.npz") != metadata["mask_cache_sha256"]:
        raise ValueError("自动分类掩模缓存发生变化。")
    expected = set(read_cohort(cfg).query("split == 'validation'").mr)
    if set(metadata["predictions_sha256"]) != expected:
        raise ValueError("自动掩模缓存的验证患者不一致。")
    for mr, checksum in metadata["predictions_sha256"].items():
        if digest(loc["validation"] / f"{mr}.nii.gz") != checksum:
            raise ValueError(f"{mr}的原生自动预测与四层缓存不一致，请使用新实验版本。")


## 1. 核对标签和训练配置

先查看三类病灶的名称、标注范围、nnU-Net 数据集编号、训练维度和设备。标签名称应与人工标注规则一致，后续数据集说明和结果表都使用这套对应关系。


In [ ]:
print("标签定义：", cfg["label_names"])
print("标注范围：", cfg["annotation_scope"])
print("配置：", cfg["segmentation"])
print("输出位置：", locations(cfg))


## 2. 导出 nnU-Net 数据集

每位患者的 T1-FLAIR、T2、DWI 分别导出为编号 `0000`、`0001`、`0002` 的三个通道。开发患者的影像和人工标签放入 `imagesTr`、`labelsTr`；时间验证影像放入 `imagesTs`，验证真值仍从患者原目录读取。

程序还会保存固定的五折划分，并为每折单独整理留出患者的预测输入。nnU-Net 折号从 0 开始，对应患者清单中从 1 开始的开发折号。导出采用文件复制，需要为三维影像及各折预测输入预留存储空间。


In [ ]:
RUN_EXPORT = False

if RUN_EXPORT:
    print(export_dataset(cfg))


## 3. 查看各阶段将执行的命令

这一单元列出预处理、五折训练、开发折外预测和时间验证预测的命令，便于先核对数据集编号、训练配置、折号及输入输出路径。此处只显示命令，下一节再分别执行。


In [ ]:
import shlex
plan = command_plan(cfg)
for phase, commands in plan.items():
    print()
    print(phase)
    for command in commands:
        print(shlex.join(command))


## 4. 预处理与五折训练

先运行 nnU-Net 的数据检查、规划与预处理，让框架根据开发数据准备训练输入。预处理完成后，放入已经确定的五折划分，再开始训练。

下面三个步骤分别设置开关，可按阶段执行并查看日志。


In [ ]:
RUN_PREPROCESS = False

if RUN_PREPROCESS:
    run_phase(cfg, "preprocess")


### 放入固定的五折划分

预处理完成后，将 `splits_final.json` 放到 nnU-Net 对应的数据目录，使分割模型沿用 00 确定的开发折。如果目录中已有划分，函数会核对两者是否一致。


In [ ]:
INSTALL_SPLITS = False

if INSTALL_SPLITS:
    print(install_splits(cfg))


### 开始五折训练

按折号 0～4 依次训练，每次使用四折患者拟合模型、一折患者进行开发评价。运行时查看各折日志，训练权重和中间结果保存在 `nnUNet_results/`。


In [ ]:
RUN_SEGMENTATION_TRAINING = False
if RUN_SEGMENTATION_TRAINING:
    run_phase(cfg, "train")


## 5. 预测开发留出患者

每个折模型只预测自己对应的留出患者，合并后得到开发队列的折外分割结果。这些结果可以用于检查哪些病例或标签较难勾画，作为分割模型开发阶段的评价依据。


In [ ]:
RUN_OOF_PREDICTION = False
if RUN_OOF_PREDICTION:
    run_phase(cfg, "predict_oof")


### 评价每类病灶的分割质量

Dice 衡量预测与人工标注的重叠，HD95 衡量边界距离，后者需要影像明确使用毫米空间单位。下面同时输出逐病例记录和按标签汇总的均值。

| 前景情况 | 指标处理 |
| --- | --- |
| 预测与人工标注均有前景 | 计算 Dice 和 HD95 |
| 只有一方有前景 | Dice 记为 0，HD95 留空并记录状态 |
| 双方均无前景 | 两项均留空，单列状态，不计入均值 |

阅读汇总表时，同时看有效病例数，尤其关注小病灶或低频标签。


In [ ]:
EVALUATE_DEVELOPMENT_SEGMENTATION = False

if EVALUATE_DEVELOPMENT_SEGMENTATION:
    cases, summary = evaluate_masks(cfg, scope="development_oof")
    display(summary)
    display(cases.head())


## 6. 预测时间验证患者

分割配置确定后，用五个折模型集成预测 `imagesTs` 中的时间验证影像，结果写入 `validation_predictions/`。06 会读取这些文件，计算验证分割指标，并可进一步运行自动掩模替换实验。

配置选择和后处理方案应在开发阶段确定，时间验证用于评价确定后的方案。


In [ ]:
RUN_VALIDATION_MASK_PREDICTION = False
if RUN_VALIDATION_MASK_PREDICTION:
    run_phase(cfg, "predict_validation")


## 自动掩模怎样接入分类模型

06 会按分类缓存中记录的四个层面，从自动分割结果中提取标签 1、2、3，并缩放成分类器需要的输入。分类器仍是 02 中用人工掩模训练的模型，因此这一步主要观察它对掩模来源变化的敏感程度；取层仍依赖 Excel 中的人工记录。

若后续要训练完整的自动勾画与分类流程，需要让分类器学习自动掩模，并按分类外层折重新安排分割训练，确保用于评估的患者没有参与对应的分割模型训练。

数据组织和运行方式可参考 nnU-Net 的[数据格式](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/reference/dataset-format.md)、[训练](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/how-to/train-models.md)及[推理说明](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/how-to/run-inference.md)。
